# Layer 2 — Data Pre-processing dan Association Rule Mining

Notebook ini menyiapkan aturan asosiasi dari riwayat belanja (`eval_set = prior`) untuk dipakai sebagai masukan fitur pada Layer 3.

Alur memorinya dijaga di empat titik:

1. `orders.csv` dibaca hanya pada kolom yang diperlukan. Pada laptop, diambil **50.000 `order_id` prior pertama**. Pada server kampus (`RUN_ON_SERVER = True`), seluruh pesanan `prior` dipakai tanpa sampling.
2. `order_products__prior.csv` (32,4 juta baris) dibaca per batch. Baris di luar populasi yang dipilih langsung dibuang.
3. Produk dengan support di bawah ambang dibuang **sebelum** one-hot encoding. Pada Apriori, item di bawah `min_support` tidak mungkin masuk itemset yang lolos, jadi langkah ini tidak mengubah hasil, tetapi mencegah matriks selebar puluhan ribu kolom.
4. Matriks keranjang disimpan sebagai boolean, lalu `apriori(..., low_memory=True)` dijalankan.

Parameter mengikuti flag `RUN_ON_SERVER` di sel konfigurasi:

| Parameter | Laptop (`False`) | Server (`True`) |
|---|---|---|
| `SAMPLE_SIZE` | 50.000 | `None` (seluruh pesanan prior) |
| `MIN_SUPPORT` | 0,01 | 0,005 |
| `min_confidence` | 0,1 | 0,1 |
| Keluaran | `outputs/apriori_rules.csv` | `outputs/apriori_rules.csv` |

In [1]:
# ==========================================
# ENVIRONMENT CONFIGURATION
# Ubah menjadi True jika dijalankan di Server Kampus (RAM/CPU besar)
# Ubah menjadi False jika dijalankan di Laptop Lokal
# ==========================================
RUN_ON_SERVER = False

from pathlib import Path
import gc
import inspect

import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

def find_project_dir() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset" / "orders.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Folder proyek tidak ditemukan. "
        "Buka Jupyter dari folder Cross Selling Retail."
    )

PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "dataset"
OUTPUT_PATH = PROJECT_DIR / "outputs" / "apriori_rules.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

MIN_CONFIDENCE = 0.1
CHUNK_SIZE = 1_000_000
TOP_N = 15

if RUN_ON_SERVER:
    # None = jangan disampel; seluruh pesanan prior di orders.csv ikut ditambang.
    SAMPLE_SIZE = None
    MIN_SUPPORT = 0.005
    print("--> [INFO] Berjalan dalam mode SERVER (Parameter Maksimal)")
else:
    SAMPLE_SIZE = 50_000
    MIN_SUPPORT = 0.01
    print("--> [INFO] Berjalan dalam mode LAPTOP (Parameter Terbatas)")

print(
    "--> [INFO] Parameter Layer 2 siap: "
    f"SAMPLE_SIZE={SAMPLE_SIZE}, min_support={MIN_SUPPORT}, min_confidence={MIN_CONFIDENCE}."
)
print(f"--> [INFO] Folder proyek: {PROJECT_DIR}")

## 1. Sampling pesanan prior

`orders.csv` tersusun per `user_id`. Pada laptop, 50.000 pesanan prior pertama adalah pesanan milik pengguna di awal berkas, bukan sampel acak seluruh populasi. Pada server (`SAMPLE_SIZE = None`), seluruh pesanan `eval_set = prior` dipakai. Pesanan `train` dan `test` tidak masuk populasi Apriori karena isi keranjang historisnya tidak ada di `order_products__prior.csv`.

In [2]:
orders = pd.read_csv(DATA_DIR / "orders.csv", usecols=["order_id", "eval_set"])
prior_order_ids = (
    orders.loc[orders["eval_set"].eq("prior"), "order_id"]
    .drop_duplicates()
)

if SAMPLE_SIZE is None:
    print("--> [INFO] Memakai seluruh order_id prior dari orders.csv, tanpa sampling...")
    sample_order_ids = prior_order_ids.to_numpy()
else:
    print(f"--> [INFO] Memulai sampling {SAMPLE_SIZE:,} order_id prior pertama dari orders.csv...")
    sample_order_ids = prior_order_ids.head(SAMPLE_SIZE).to_numpy()
sample_id_set = set(sample_order_ids.tolist())

print(f"Pesanan prior dalam sampel: {len(sample_order_ids):,}")
print(f"order_id pertama: {sample_order_ids[0]:,} | terakhir: {sample_order_ids[-1]:,}")

del orders
gc.collect()

Pesanan prior dalam sampel: 50,000
order_id pertama: 2,539,329 | terakhir: 2,154,808


3

## 2. Penggabungan transaksi dan nama produk

Hanya `order_id` dan `product_id` yang diambil dari file prior. Setelah batch digabung, nama produk disambungkan dari `products.csv`. Spasi tak-terputus (`\xa0`) di akhir empat nama produk dibersihkan supaya kolom one-hot tidak terpecah.

In [3]:
print("--> [INFO] Memindai order_products__prior.csv per batch dan menggabungkan nama produk...")
parts = []
for chunk in pd.read_csv(
    DATA_DIR / "order_products__prior.csv",
    usecols=["order_id", "product_id"],
    chunksize=CHUNK_SIZE,
):
    hit = chunk.loc[chunk["order_id"].isin(sample_id_set), ["order_id", "product_id"]]
    if not hit.empty:
        parts.append(hit)

line_items = pd.concat(parts, ignore_index=True)
del parts
gc.collect()

products = pd.read_csv(DATA_DIR / "products.csv", usecols=["product_id", "product_name"])
products["product_name"] = (
    products["product_name"].str.replace("\xa0", " ", regex=False).str.strip()
)
if products["product_name"].duplicated().any():
    raise ValueError("Nama produk tidak unik setelah pembersihan spasi.")

basket_df = line_items.merge(products, on="product_id", how="left")
basket_df = basket_df.loc[:, ["order_id", "product_id", "product_name"]]

if basket_df["product_name"].isna().any():
    raise ValueError("Ada product_id sampel yang tidak ditemukan di products.csv.")

print(f"Baris item dalam sampel: {len(basket_df):,}")
print(f"Pesanan yang punya item: {basket_df['order_id'].nunique():,}")
print(f"Produk unik sebelum saringan support: {basket_df['product_name'].nunique():,}")
basket_df.head()

Baris item dalam sampel: 496,364
Pesanan yang punya item: 50,000
Produk unik sebelum saringan support: 24,998


,order_id,product_id,product_name
0,8,23423,Original Hawaiian Sweet Rolls
1,40,10070,Organic 1% Low Fat Milk
2,40,42450,Macaroni & Cheese
3,40,33198,Sparkling Natural Mineral Water
4,40,34866,Chocolate Milk 1% Milkfat


## 3. Matriks keranjang (one-hot)

Support dihitung sebagai jumlah pesanan yang memuat produk dibagi 50.000. Produk di bawah `min_support` dibuang, lalu sisa item diubah menjadi matriks:

- baris = `order_id` sampel, termasuk pesanan yang tidak lagi punya produk setelah saringan;
- kolom = `product_name`;
- nilai = `True` jika produk ada di pesanan tersebut.

Pesanan kosong tetap dipertahankan supaya penyebut support tetap 50.000.

In [4]:
print("--> [INFO] Menyaring produk di bawah min_support dan membentuk matriks keranjang one-hot...")
n_orders = len(sample_order_ids)
order_support = basket_df.groupby("product_name")["order_id"].nunique()
frequent_products = order_support[(order_support / n_orders) >= MIN_SUPPORT].index

basket_frequent = (
    basket_df.loc[basket_df["product_name"].isin(frequent_products), ["order_id", "product_name"]]
    .drop_duplicates()
    .assign(bought=True)
)

basket_matrix = (
    basket_frequent
    .pivot_table(
        index="order_id",
        columns="product_name",
        values="bought",
        aggfunc="max",
        fill_value=False,
    )
    .reindex(sample_order_ids, fill_value=False)
    .astype(bool)
)
basket_matrix.index.name = "order_id"

print(f"Produk lolos min_support {MIN_SUPPORT}: {basket_matrix.shape[1]:,}")
print(f"Bentuk matriks: {basket_matrix.shape[0]:,} pesanan x {basket_matrix.shape[1]:,} produk")
print(f"Memori matriks: {basket_matrix.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

del basket_df, basket_frequent, order_support
gc.collect()

basket_matrix.iloc[:5, :8]

Produk lolos min_support 0.01: 102
Bentuk matriks: 50,000 pesanan x 102 produk
Memori matriks: 5.2 MB


product_name,100% Whole Wheat Bread,Apple Honeycrisp Organic,Asparagus,Bag of Organic Bananas,Banana,Bartlett Pears,Blueberries,Boneless Skinless Chicken Breasts
order_id,,,,,,,,
2539329,False,False,False,False,False,False,False,False
2398795,False,False,False,True,False,False,False,False
473747,False,False,False,False,False,False,False,False
2254736,False,False,False,False,False,False,False,False
431534,False,False,False,True,False,True,False,False


## 4. Apriori dan aturan asosiasi

`low_memory=True` membangun kandidat itemset secara bertahap. Aturan dipertahankan bila confidence minimal 0,1. Dataframe lengkap disimpan ke `outputs/apriori_rules.csv`, terurut dari confidence lalu lift.

In [5]:
print("--> [INFO] Menjalankan Apriori dan mengekstrak association rules...")
frequent_itemsets = apriori(
    basket_matrix,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    low_memory=True,
)

print(f"Frequent itemset: {len(frequent_itemsets):,}")
print(frequent_itemsets["itemsets"].map(len).value_counts().sort_index().rename("jumlah").to_frame())

rule_kwargs = {
    "metric": "confidence",
    "min_threshold": MIN_CONFIDENCE,
}
if "num_itemsets" in inspect.signature(association_rules).parameters:
    # Wajib pada mlxtend 0.23+: jumlah transaksi sumber untuk metrik aturan.
    rule_kwargs["num_itemsets"] = n_orders

rules = association_rules(frequent_itemsets, **rule_kwargs)

def format_itemset(itemset):
    return " + ".join(sorted(itemset))

apriori_rules = rules.copy()
apriori_rules["antecedents"] = apriori_rules["antecedents"].map(format_itemset)
apriori_rules["consequents"] = apriori_rules["consequents"].map(format_itemset)
apriori_rules = apriori_rules.sort_values(
    ["confidence", "lift", "support"],
    ascending=False,
).reset_index(drop=True)

apriori_rules.to_csv(OUTPUT_PATH, index=False)
print(f"Aturan tersimpan: {len(apriori_rules):,} -> {OUTPUT_PATH.name}")

display_cols = [
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift",
]

print(f"\n{TOP_N} aturan teratas menurut confidence")
display(apriori_rules.loc[:, display_cols].head(TOP_N))

print(f"\n{TOP_N} aturan teratas menurut lift")
display(
    apriori_rules.sort_values(["lift", "confidence", "support"], ascending=False)
    .loc[:, display_cols]
    .head(TOP_N)
    .reset_index(drop=True)
)

Frequent itemset: 116
          jumlah
itemsets        
1            102
2             14
Aturan tersimpan: 24 -> apriori_rules.csv

15 aturan teratas menurut confidence


,antecedents,consequents,support,confidence,lift
0,Organic Raspberries,Bag of Organic Bananas,0.01362,0.315424,2.608102
1,Cucumber Kirby,Banana,0.01008,0.309392,2.266278
2,Organic Hass Avocado,Bag of Organic Bananas,0.01992,0.309125,2.556018
3,Organic Avocado,Banana,0.01602,0.307840,2.254909
4,Strawberries,Banana,0.01192,0.282063,2.066096
5,Organic Raspberries,Organic Strawberries,0.01192,0.276054,3.276991
6,Large Lemon,Banana,0.01350,0.267221,1.957375
7,Organic Whole Milk,Bag of Organic Bananas,0.01180,0.253002,2.091961
8,Organic Strawberries,Bag of Organic Bananas,0.02078,0.246676,2.039657
9,Organic Baby Spinach,Banana,0.01502,0.208033,1.523830



15 aturan teratas menurut lift


,antecedents,consequents,support,confidence,lift
0,Organic Raspberries,Organic Strawberries,0.01192,0.276054,3.276991
1,Organic Strawberries,Organic Raspberries,0.01192,0.141500,3.276991
2,Organic Raspberries,Bag of Organic Bananas,0.01362,0.315424,2.608102
3,Bag of Organic Bananas,Organic Raspberries,0.01362,0.112618,2.608102
4,Bag of Organic Bananas,Organic Hass Avocado,0.01992,0.164710,2.556018
5,Organic Hass Avocado,Bag of Organic Bananas,0.01992,0.309125,2.556018
6,Organic Hass Avocado,Organic Strawberries,0.01318,0.204531,2.427960
7,Organic Strawberries,Organic Hass Avocado,0.01318,0.156458,2.427960
8,Cucumber Kirby,Banana,0.01008,0.309392,2.266278
9,Organic Avocado,Banana,0.01602,0.307840,2.254909
